This R script represents a simple LLM-based agentic workflow which **extracts, tabulates and QC PK information from the FDA labels**. The key steps are:
- retrieval of relevant labels via OpenFDA API;
- LLM-based extraction of PK information;
- Two level LLM-based QC of extracted PK information:
-- check of tabulated PK line-by-line;
-- chcek at for missing infomation at the compound level;

The script uses OpenAI gpt-4o LLM, therefore OPENAI_API_KEY needs to provided, alternatively a different LLM can be used.

Make sure that run time is set to R:
Runtime->Change Runtime type-> Runtime = R



In [ ]:
# set up env
rm(list=ls())

if (!requireNamespace("ellmer", quietly = TRUE)) {
  install.packages("ellmer")
}
library(ellmer) # R interface to LLMs

if (!requireNamespace("httr2", quietly = TRUE)) {
  install.packages("httr2")
}
library(httr2) # supports API usage in R

if (!requireNamespace("jsonlite", quietly = TRUE)) {
  install.packages("jsonlite")
}
library(jsonlite) # JSON Parser for R

# below are standard R data wrangling packages
if (!requireNamespace("dplyr", quietly = TRUE)) {
  install.packages("dplyr")
}
library(dplyr)

if (!requireNamespace("readr", quietly = TRUE)) {
  install.packages("readr")
}
library(readr)

if (!requireNamespace("stringr", quietly = TRUE)) {
  install.packages("stringr")
}
library(stringr)

if (!requireNamespace("tibble", quietly = TRUE)) {
  install.packages("tibble")
}
library(tibble)

if (!requireNamespace("ggplot2", quietly = TRUE)) {
  install.packages("ggplot2")
}
library(ggplot2)

# a simple test of R environment
# a scatter plot shold generated
iris %>%
ggplot(aes(Sepal.Length, Sepal.Width, color=Species))+
geom_point(alpha=0.7)

In [1]:
#your OPENAI API KEY is confidential
OPENAI_API_KEY <- "YOUR_OPENAI_API_KEY"

In [ ]:
# test chatgpt access
message = "This a simple test."
chat <- chat_openai(message, model = "gpt-4o", api_key = OPENAI_API_KEY )
print(chat)
chat$chat("This is a test")


In [ ]:
# prompts
message_system   ="You an experienced pharmacokineticist with attention to the detail. You specialize in extracting PK parameters from the text. You always return results in JSON format."
message_user     ="The following text was extracted from FDA labelling document. Extract reported PK parameters e.g. CL (clearance), Vd (volume of distribution), AUC (Caverage), Cmin (Ctrough), Cmax also extract uncertainty of the estimate as well as units. Extracted parameters should be reported in tabular view. Table columns should include: drug, dose, population, parameter, value, uncertainty_value, uncertainty_type, unit."
message_user_QC  ="The following pharmacokinetic parameter and metadata were extracted from the PK section of the FDA labelling document. Perform quality control of the extracted PK parameter. Specifically, confirm numerical accuracy of the extracted PK parameter, its uncertainty and metadata describing unit of the parameter value, dose and population it was extracted from. Response should include two and only two values: QC: 1 = passed, 0 = failed. QC_comment: empty string if QC=1, a short description of identified error if QC=0."
message_user_QC2 ="The following summary table in JSON format contains pharmacokinetic parameters and metadata extracted from the PK section of the FDA labelling document. Check if all PK parameters, for which numerical values are available in the FDA labelling document, specifically CL (clearance), Vd (volume of distribution), AUC (Caverage), Cmin (Ctrough), Cmax are present in the summary table. If an additional PK parameter is mentioned, however it's numerical value is not reported - ignore it. Response should include two and only two values COMPLETE and COMMENT. COMPLETE: is set to 1 if summary table is complete and to 0 if summary table is not complete. COMMENT: contains empty string if COMPLETE=1, if COMPLETE=0 COMMENT lists PK parameters missing in the summary table, specifically PK parameters, their values and population. Importantly, COMPLETE should be a single string."

In [ ]:
# Retrieve PK data from OpenFDA API
# 3 labels containing via pediatric PK information will be retrieved
drug_query <- "*mab+AND+pharmacokinetics:pediat*"
url <- paste0("https://api.fda.gov/drug/label.json?search=openfda.generic_name:", drug_query, "&limit=3")

req <- request(url) |> req_perform()
if (resp_status(req) == 200) {
  data <- resp_body_json(req)
  print("API request successful!")
} else {
  stop("API request failed")
}

In [ ]:
# helper function
# safely converts json to data frame
# code was generated with Gemini 3 Falsh

parse_llm_response <- function(raw_response) {
  if (!is.character(raw_response) ||
      length(raw_response) != 1L ||
      is.na(raw_response) ||
      !nzchar(trimws(raw_response))) {
    warning("The LLM response is empty.")
    return(NULL)
  }

  # Remove whitespace and Markdown code fences such as:
  # ```json
  # [...]
  # ```
  clean_json <- raw_response |>
    str_trim() |>
    str_replace(
      regex("^```json\\s*", ignore_case = TRUE),
      ""
    ) |>
    str_replace(
      regex("^```\\s*", ignore_case = TRUE),
      ""
    ) |>
    str_replace(
      regex("\\s*```$"),
      ""
    ) |>
    str_trim()

  # Parse JSON safely
  parsed_data <- tryCatch(
    {
      fromJSON(
        clean_json,
        simplifyDataFrame = TRUE,
        flatten = TRUE
      )
    },
    error = function(e) {
      warning(
        "Could not parse the LLM response as JSON: ",
        conditionMessage(e)
      )
      return(NULL)
    }
  )

  if (is.null(parsed_data)) {
    return(NULL)
  }

  # Handle an empty JSON array: []
  if (is.data.frame(parsed_data) && nrow(parsed_data) == 0L) {
    message("Empty JSON.")
    return(NULL)
  }

  # Handle a single JSON object:
  # {"drug": "...", "parameter": "..."}
  if (is.list(parsed_data) && !is.data.frame(parsed_data)) {
    parsed_data <- as.data.frame(
      parsed_data,
      stringsAsFactors = FALSE
    )
  }

  temp_df <- as_tibble(parsed_data)
  #print(temp_df)

  return( temp_df )
}


append_drug_names_f <- function( temp_df,
                                 brand_name,
                                 generic_name )
{

  temp_df <- temp_df |>
    mutate(
      source_brand_name   = brand_name,
      source_generic_name = generic_name
    )


  #print( temp_df )
  message(
    sprintf(
      "Successfully extracted %d parameters.",
      nrow(temp_df)
    )
  )

  return( temp_df  )
}

# combine functions above
parse_pk_response <- function(raw_response,
                              brand_name,
                              generic_name){

  temp_df <- parse_llm_response(raw_response)
  resp    <- append_drug_names_f( temp_df, brand_name, generic_name )
  return( resp )

 }

In [ ]:
# helper function to do QC
do_qc_f <- function( df_pk_cur, pk_text ){

  chat_QC <- chat_openai(
      system_prompt = message_system,
      model = "gpt-4o",
      api_key = OPENAI_API_KEY,
      echo = "none"
    )

    json_pk1 <-  df_pk_cur %>% as.list( ) %>%
      jsonlite::toJSON( auto_unbox = TRUE, pretty = TRUE )
    json_pk1_str <- gsub("^\\[|\\]$", "", json_pk1)

  # Create prompt by combining user instruction with the pk section and
  # extracted PK parameters
  updated_message_user_QC <- paste0(
    message_user_QC,
    ". Extracted PK parameters in JSON format: ",
    json_pk1_str,
    ". PK Section of the labelling document: ",
    pk_text
  )

  # Submit prompt and get response from the LLM
  raw_response_QC <- chat_QC$chat(updated_message_user_QC)

}

# helper function completness check
check_completness_f <- function( temp_df , pk_text ){

  chat_QC <- chat_openai(
    system_prompt = message_system,
    model = "gpt-4o",
    api_key = OPENAI_API_KEY,
    echo = "none"
  )

   json_pk1 <-  temp_df  %>%
     select(c("dose", "population", "parameter", "value" ))   %>% t() %>%
     as.data.frame() %>%      as.list( ) %>%
      jsonlite::toJSON( auto_unbox = TRUE, pretty = TRUE )
    json_pk1_str <- gsub("^\\[|\\]$", "", json_pk1)
  #print( json_pk1_str  )

  # Create prompt by combining user instruction with the pk section
  # Removed cat() and added missing comma before pk_text
  updated_message_user_QC2 <- paste0(
    message_user_QC2,
    " Extracted PK parameters in JSON format: ",
    json_pk1_str,
    " PK Section of the labelling document: ",
    pk_text
  )
  #print( updated_message_user_QC2 )

  # Submit prompt and get response from the LLM
  raw_response_QC <- chat_QC$chat(updated_message_user_QC2)

  # Display the QC result
  cat(raw_response_QC)

  return(  as.character(raw_response_QC) )

}

In [ ]:
# run analysis
all_pk_dfs <- list()
qc_missing <- list()
results <- data[[2]]

# iterate through the list of results returned by OpenFDA compound by compound
for (i in seq_along(results)) {
  res <- results[[i]]
  generic_name <- res$openfda$generic_name[[1]] %||% "Unknown"
  brand_name   <- res$openfda$brand_name[[1]] %||% "Unknown"

  # Extract PK text
  # TODO, merge section 12
  pk_text <- paste(res$pharmacokinetics, collapse = " ")

  if (nchar(pk_text) < 10) next

  message(paste("--- Processing:", generic_name, "---"))

  # Create a new chat object which enables interaction with LLM
  # Creating a new chat object in a loop makes each function call stateless.
  # This is an important difference with Python.
  # In Python interaction with LLM is by default is stateless
  chat <- chat_openai(
    system_prompt = message_system,
    model = "gpt-4o",
     api_key = OPENAI_API_KEY,
    echo = "none"
  )

  # create prompt by combining user instruction with the pk section
  updated_message_user = paste0( message_user, " ",  pk_text)

  # submit prompt and get response from the LLM
  raw_response <- chat$chat( updated_message_user )

  # parse LLM response and return tibble
  temp_df <- parse_pk_response(raw_response,
                               brand_name,
                               generic_name) %>%
        mutate(across(everything(), as.character))

  # extracted PK parameters
  print( temp_df )

  # copmletness check
  print("Run completness check")
  qc_cur <- check_completness_f( temp_df , pk_text ) %>%
              parse_llm_response( )

  qc_cur <- qc_cur %>%
    mutate(
      source_brand_name   = brand_name,
      source_generic_name = generic_name
    )

  print("Run line-by-line QC")

  # QC temp_df line-by line
#if (FALSE){
  all_pk_qc <- list()
  for (j in 1:nrow(temp_df) ){
    df_pk_cur <- temp_df[j,]
    raw_qc    <- do_qc_f(df_pk_cur, pk_text )
    temp_qc   <- raw_qc %>% parse_llm_response()

    if (!is.null(temp_qc) && nrow(temp_qc) > 0L) {
    all_pk_qc[[length(all_pk_qc) + 1L]] <- temp_qc
    }
  }
 long_df_qc <- bind_rows(all_pk_qc)
 temp_df    <- cbind( temp_df, long_df_qc ) %>%
    mutate(across(everything(), as.character))
 #}

  if (!is.null(temp_df) && nrow(temp_df) > 0L) {
  all_pk_dfs[[length(all_pk_dfs) + 1L]] <- temp_df
  qc_missing[[length(qc_missing) + 1L]] <- qc_cur
   }

}

# combine all tibles into one long table
final_long_df <- bind_rows(all_pk_dfs)
final_long_qc <- bind_rows(qc_missing)
print(final_long_df)
print(qc_missing)

# Save result
write_csv(final_long_df, "Extracted_PK_QC_R.csv")
write_csv(final_long_qc, "QC_completness_R.csv")